In [10]:
import numpy as np
import pywt
import os
from PIL import Image
from scipy.fftpack import dct
from scipy.fftpack import idct
import pathlib

# This gets the current working directory, which is usually the folder the notebook is in
current_path = str(pathlib.Path().resolve())

# FIX 1: Change image and watermark paths to simple filenames
# (assuming the files 'imagetest1.jpg' and 'qrcodetest1.png' are in the current working directory, /content/)
image = 'images.png'
watermark = 'nature.jpg'

# FIX 2: Create necessary directories for I/O
os.makedirs('./pictures', exist_ok=True)
os.makedirs('./dataset', exist_ok=True)
os.makedirs('./result', exist_ok=True)

def convert_image(image_name, size):
    # FIX 3: If the image is in the current directory, open it directly by name.
    # If the image is intended to be moved INTO './pictures/', you need a separate step.
    # Based on the original code's logic, it seems 'image_name' is the full input file.

    # We'll assume for simplicity that the original files are in the *root* # and we want to process them directly.

    # OLD INCORRECT LINE: img = Image.open('./pictures/' + image_name).resize((size, size), 1)

    # Corrected path to open the source image
    img = Image.open(image_name).resize((size, size), Image.Resampling.LANCZOS) # Use Resampling for modern Pillow
    img = img.convert('L')

    # The output is saved to the dataset folder
    img.save('./dataset/' + image_name)

    # Note: np.float is deprecated in recent numpy. Use np.float64
    image_array = np.array(img.getdata(), dtype=np.float64).reshape((size, size))
    print(image_array[0][0])
    print(image_array[10][10])

    return image_array

def process_coefficients(imArray, model, level):
    coeffs=pywt.wavedec2(data = imArray, wavelet = model, level = level)
    # print coeffs[0].__len__()
    coeffs_H=list(coeffs)

    return coeffs_H


def embed_mod2(coeff_image, coeff_watermark, offset=0):
    # Note: xrange is Python 2. Use range in Python 3
    for i in range(len(coeff_watermark)):
        for j in range(len(coeff_watermark[i])):
            coeff_image[i*2+offset][j*2+offset] = coeff_watermark[i][j]

    return coeff_image

def embed_mod4(coeff_image, coeff_watermark):
    # Note: xrange is Python 2. Use range in Python 3
    for i in range(len(coeff_watermark)):
        for j in range(len(coeff_watermark[i])):
            coeff_image[i*4][j*4] = coeff_watermark[i][j]

    return coeff_image



def embed_watermark(watermark_array, orig_image):
    watermark_flat = watermark_array.ravel()
    ind = 0

    # Using len(orig_image) is correct for the size
    size = len(orig_image)
    for x in range (0, size, 8):
        for y in range (0, size, 8):
            if ind < len(watermark_flat):
                subdct = orig_image[x:x+8, y:y+8]
                subdct[5][5] = watermark_flat[ind]
                orig_image[x:x+8, y:y+8] = subdct
                ind += 1


    return orig_image



def apply_dct(image_array):
    size = len(image_array)
    all_subdct = np.empty((size, size))
    for i in range (0, size, 8):
        for j in range (0, size, 8):
            subpixels = image_array[i:i+8, j:j+8]
            subdct = dct(dct(subpixels.T, norm="ortho").T, norm="ortho")
            all_subdct[i:i+8, j:j+8] = subdct

    return all_subdct


def inverse_dct(all_subdct):
    size = len(all_subdct)
    all_subidct = np.empty((size, size))
    for i in range (0, size, 8):
        for j in range (0, size, 8):
            subidct = idct(idct(all_subdct[i:i+8, j:j+8].T, norm="ortho").T, norm="ortho")
            all_subidct[i:i+8, j:j+8] = subidct

    return all_subidct


def get_watermark(dct_watermarked_coeff, watermark_size):

    subwatermarks = []

    size = len(dct_watermarked_coeff)
    for x in range (0, size, 8):
        for y in range (0, size, 8):
            coeff_slice = dct_watermarked_coeff[x:x+8, y:y+8]
            subwatermarks.append(coeff_slice[5][5])

    watermark = np.array(subwatermarks).reshape(watermark_size, watermark_size)

    return watermark


def recover_watermark(image_array, model='haar', level = 1):


    coeffs_watermarked_image = process_coefficients(image_array, model, level=level)
    dct_watermarked_coeff = apply_dct(coeffs_watermarked_image[0])

    watermark_array = get_watermark(dct_watermarked_coeff, 128)

    watermark_array = np.uint8(watermark_array)

#Save result
    img = Image.fromarray(watermark_array)
    img.save('./result/recovered_watermark.jpg')


def print_image_from_array(image_array, name):

    image_array_copy = image_array.clip(0, 255)
    image_array_copy = image_array_copy.astype("uint8")
    img = Image.fromarray(image_array_copy)
    img.save('./result/' + name)



def w2d(img):
    model = 'haar'
    level = 1

    # Pass the filename directly
    image_array = convert_image(image, 2048)
    watermark_array = convert_image(watermark, 128)

    coeffs_image = process_coefficients(image_array, model, level=level)
    dct_array = apply_dct(coeffs_image[0])
    dct_array = embed_watermark(watermark_array, dct_array)
    coeffs_image[0] = inverse_dct(dct_array)


# reconstruction
    image_array_H=pywt.waverec2(coeffs_image, model)
    print_image_from_array(image_array_H, 'image_with_watermark.jpg')



# recover images
    recover_watermark(image_array = image_array_H, model=model, level = level)


w2d("test")

255.0
255.0
94.0
101.0


In [2]:
!pip install numpy PyWavelets Pillow scipy

In [12]:
import numpy as np
import pywt
import os
from PIL import Image
from scipy.fftpack import dct, idct
# REMOVED: from skimage.metrics import normalized_cross_correlation as NCC
from scipy.ndimage import gaussian_filter

# =========================================================================
# === 1. IMPORTS & SETUP (from your original code)
# =========================================================================

# ... (rest of your initial setup and helper functions)

# --- Watermarking & Recovery Functions (Minimal set for this script) ---

# All your original functions go here (e.g., get_watermark, apply_dct, process_coefficients, recover_watermark)
# Note: I am including the definitions again for a self-contained, working solution.

WATERMARK_SIZE = 128
os.makedirs('./attacks', exist_ok=True) # Directory for saving attacked images

def get_watermark(dct_watermarked_coeff, watermark_size):
    subwatermarks = []
    size = len(dct_watermarked_coeff)
    for x in range (0, size, 8):
        for y in range (0, size, 8):
            coeff_slice = dct_watermarked_coeff[x:x+8, y:y+8]
            subwatermarks.append(coeff_slice[5][5])
    watermark = np.array(subwatermarks).reshape(watermark_size, watermark_size)
    return watermark

def apply_dct(image_array):
    size = len(image_array)
    all_subdct = np.empty((size, size))
    for i in range (0, size, 8):
        for j in range (0, size, 8):
            subpixels = image_array[i:i+8, j:j+8]
            subdct = dct(dct(subpixels.T, norm="ortho").T, norm="ortho")
            all_subdct[i:i+8, j:j+8] = subdct
    return all_subdct

def process_coefficients(imArray, model, level):
    coeffs=pywt.wavedec2(data = imArray, wavelet = model, level = level)
    coeffs_H=list(coeffs)
    return coeffs_H


def recover_watermark(image_array, model='haar', level = 1):
    coeffs_watermarked_image = process_coefficients(image_array, model, level=level)
    dct_watermarked_coeff = apply_dct(coeffs_watermarked_image[0])
    watermark_array = get_watermark(dct_watermarked_coeff, WATERMARK_SIZE)
    return watermark_array

# ... (The rest of your utility functions for attacks)

def attack_jpeg_compression(image_array, quality):
    """Simulates JPEG compression attack."""
    img = Image.fromarray(np.uint8(image_array))
    attack_path = f'./attacks/jpeg_q{quality}.jpg'
    img.save(attack_path, 'jpeg', quality=quality)
    attacked_img = Image.open(attack_path).convert('L')
    return np.array(attacked_img.getdata()).reshape(image_array.shape)

def attack_gaussian_noise(image_array, sigma):
    """Adds Gaussian (Random) Noise."""
    normalized_array = image_array / 255.0
    noise = np.random.normal(0, sigma, image_array.shape)
    attacked_array = normalized_array + noise
    attacked_array = np.clip(attacked_array, 0, 1) * 255.0
    Image.fromarray(np.uint8(attacked_array)).save(f'./attacks/noise_s{sigma:.2f}.jpg')
    return attacked_array

def attack_gaussian_blur(image_array, sigma):
    """Applies Gaussian Blur (Low-Pass Filter)."""
    attacked_array = gaussian_filter(image_array, sigma=sigma)
    Image.fromarray(np.uint8(attacked_array)).save(f'./attacks/blur_s{sigma:.1f}.jpg')
    return attacked_array

# =========================================================================
# === 3. METRIC AND REPORTING (FIXED)
# =========================================================================

def measure_ncc(original_wm, recovered_wm):
    """
    Calculates the Normalized Cross-Correlation (NCC) using the standard formula:
    NCC = sum((W - mean(W)) * (W' - mean(W')) / (std(W) * std(W') * N)
    This is equivalent to the Pearson correlation coefficient between the two arrays.
    """
    # Flatten arrays
    W = original_wm.flatten()
    W_prime = recovered_wm.flatten()

    # Calculate Pearson's r, which is the definition of NCC for these arrays
    # np.corrcoef returns a 2x2 matrix, we want the off-diagonal element
    ncc_value = np.corrcoef(W, W_prime)[0, 1]

    return ncc_value

def load_grayscale_image(path, size=None):
    """Helper function to load an image as a grayscale NumPy array."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}. Please run the watermarking script first.")

    img = Image.open(path).convert('L')
    if size:
        # Use Image.Resampling.LANCZOS for quality resizing
        img = img.resize((size, size), Image.Resampling.LANCZOS)

    return np.array(img.getdata()).reshape(img.size[::-1])

def run_robustness_test():
    """Runs a battery of tests to assess watermarking robustness."""
    print("--- ROBUSTNESS ASSESSMENT STARTING ---")

    try:
        watermarked_image_path = './result/image_with_watermark.jpg'
        original_watermark_path = './dataset/nature.jpg'

        original_watermark_arr = load_grayscale_image(original_watermark_path, size=WATERMARK_SIZE)
        wm_image_arr = load_grayscale_image(watermarked_image_path)

    except FileNotFoundError as e:
        print(f"ERROR: {e}")
        print("Please ensure the watermarking process has been run successfully and the output files exist.")
        return

    test_results = {}

    print(f"\nHost Image Size: {wm_image_arr.shape}")
    print(f"Watermark Size: {original_watermark_arr.shape}")
    print("-" * 40)

    # --- A. Test Without Attack (Baseline) ---
    recovered_wm_baseline = recover_watermark(wm_image_arr)
    ncc_baseline = measure_ncc(original_watermark_arr, recovered_wm_baseline)
    test_results['Baseline'] = ncc_baseline
    print(f"| {'Baseline (No Attack):':<30} | NCC: {ncc_baseline:.4f} |")
    print("-" * 40)


    # --- B. JPEG Compression Attack ---
    print("\n--- Testing JPEG Compression (Lower Quality = Stronger Attack) ---")
    jpeg_qualities = [80, 60, 40, 20]
    for q in jpeg_qualities:
        attacked_img_arr = attack_jpeg_compression(wm_image_arr, q)
        recovered_wm = recover_watermark(attacked_img_arr)
        ncc_score = measure_ncc(original_watermark_arr, recovered_wm)
        test_results[f'JPEG Q={q}'] = ncc_score
        print(f"| {'JPEG Compression Q='+str(q):<30} | NCC: {ncc_score:.4f} |")

    # --- C. Gaussian Noise Attack ---
    print("\n--- Testing Gaussian Noise (Higher Sigma = Stronger Attack) ---")
    noise_sigmas = [0.05, 0.10, 0.20] # Standard deviation of noise (normalized)
    for s in noise_sigmas:
        attacked_img_arr = attack_gaussian_noise(wm_image_arr, s)
        recovered_wm = recover_watermark(attacked_img_arr)
        ncc_score = measure_ncc(original_watermark_arr, recovered_wm)
        test_results[f'Noise S={s:.2f}'] = ncc_score
        print(f"| {'Gaussian Noise Sigma='+f'{s:.2f}':<30} | NCC: {ncc_score:.4f} |")

    # --- D. Gaussian Blur Attack ---
    print("\n--- Testing Gaussian Blur (Higher Sigma = Stronger Attack) ---")
    blur_sigmas = [0.5, 1.0, 1.5]
    for s in blur_sigmas:
        attacked_img_arr = attack_gaussian_blur(wm_image_arr, s)
        recovered_wm = recover_watermark(attacked_img_arr)
        ncc_score = measure_ncc(original_watermark_arr, recovered_wm)
        test_results[f'Blur S={s:.1f}'] = ncc_score
        print(f"| {'Gaussian Blur Sigma='+f'{s:.1f}':<30} | NCC: {ncc_score:.4f} |")

    print("\n--- ROBUSTNESS ASSESSMENT COMPLETE ---")
    return test_results

# =========================================================================
# === 4. EXECUTION
# =========================================================================

# Run the test
robustness_results = run_robustness_test()

--- ROBUSTNESS ASSESSMENT STARTING ---

Host Image Size: (2048, 2048)
Watermark Size: (128, 128)
----------------------------------------
| Baseline (No Attack):          | NCC: 0.8678 |
----------------------------------------

--- Testing JPEG Compression (Lower Quality = Stronger Attack) ---
| JPEG Compression Q=80          | NCC: 0.8640 |
| JPEG Compression Q=60          | NCC: 0.8397 |
| JPEG Compression Q=40          | NCC: 0.8320 |
| JPEG Compression Q=20          | NCC: 0.7330 |

--- Testing Gaussian Noise (Higher Sigma = Stronger Attack) ---
| Gaussian Noise Sigma=0.05      | NCC: 0.7444 |
| Gaussian Noise Sigma=0.10      | NCC: 0.6009 |
| Gaussian Noise Sigma=0.20      | NCC: 0.4120 |

--- Testing Gaussian Blur (Higher Sigma = Stronger Attack) ---
| Gaussian Blur Sigma=0.5        | NCC: 0.8757 |
| Gaussian Blur Sigma=1.0        | NCC: 0.8896 |
| Gaussian Blur Sigma=1.5        | NCC: 0.8849 |

--- ROBUSTNESS ASSESSMENT COMPLETE ---


In [16]:
import numpy as np
import pywt
import os
from PIL import Image
from scipy.fftpack import dct, idct
from scipy.ndimage import gaussian_filter
from skimage.metrics import peak_signal_noise_ratio as PSNR

# --- CONFIGURATION ---
IMAGE_HOST = 'images.png'
IMAGE_WATERMARK = 'nature.jpg'
HOST_SIZE = 2048
WATERMARK_SIZE = 128
MODEL = 'haar'
LEVEL = 1

# --- PATHS ---
os.makedirs('./pictures', exist_ok=True)
os.makedirs('./dataset', exist_ok=True)
os.makedirs('./result', exist_ok=True)
os.makedirs('./attacks', exist_ok=True)


# =========================================================================
# === 1. CORE WATERMARKING FUNCTIONS (FROM YOUR ORIGINAL CODE)
# =========================================================================

def convert_image(image_name, size):
    """Loads image, converts to grayscale, resizes, saves to dataset, and returns array."""
    # Assuming the source image is in the current directory
    img = Image.open(image_name).resize((size, size), Image.Resampling.LANCZOS)
    img = img.convert('L')
    img.save('./dataset/' + image_name)
    image_array = np.array(img.getdata(), dtype=np.float64).reshape((size, size))
    return image_array

def process_coefficients(imArray, model, level):
    """Performs 2D DWT on the image array."""
    coeffs = pywt.wavedec2(data=imArray, wavelet=model, level=level)
    return list(coeffs)

def apply_dct(image_array):
    """Applies 8x8 block-wise DCT."""
    size = len(image_array)
    all_subdct = np.empty((size, size))
    for i in range(0, size, 8):
        for j in range(0, size, 8):
            subpixels = image_array[i:i+8, j:j+8]
            subdct = dct(dct(subpixels.T, norm="ortho").T, norm="ortho")
            all_subdct[i:i+8, j:j+8] = subdct
    return all_subdct

def inverse_dct(all_subdct):
    """Applies 8x8 block-wise Inverse DCT (IDCT)."""
    size = len(all_subdct)
    all_subidct = np.empty((size, size))
    for i in range(0, size, 8):
        for j in range(0, size, 8):
            subidct = idct(idct(all_subdct[i:i+8, j:j+8].T, norm="ortho").T, norm="ortho")
            all_subidct[i:i+8, j:j+8] = subidct
    return all_subidct

def embed_watermark(watermark_array, orig_image):
    """Embeds the flattened watermark into the (5,5) coefficient of each 8x8 DCT block."""
    watermark_flat = watermark_array.ravel()
    ind = 0
    size = len(orig_image)
    for x in range(0, size, 8):
        for y in range(0, size, 8):
            if ind < len(watermark_flat):
                # The LL sub-band array is modified in-place here
                orig_image[x+5][y+5] = watermark_flat[ind] # Modifying (5, 5) coefficient
                ind += 1
    return orig_image

def get_watermark(dct_watermarked_coeff, watermark_size):
    """Extracts the watermark from the (5,5) DCT coefficient of each 8x8 block."""
    subwatermarks = []
    size = len(dct_watermarked_coeff)
    for x in range(0, size, 8):
        for y in range(0, size, 8):
            coeff_slice = dct_watermarked_coeff[x:x+8, y:y+8]
            subwatermarks.append(coeff_slice[5][5])
    watermark = np.array(subwatermarks).reshape(watermark_size, watermark_size)
    return watermark

def print_image_from_array(image_array, name):
    """Clips array to [0, 255] and saves it as an image."""
    image_array_copy = image_array.clip(0, 255)
    image_array_copy = image_array_copy.astype("uint8")
    img = Image.fromarray(image_array_copy)
    img.save('./result/' + name)

def recover_watermark(image_array, model=MODEL, level=LEVEL):
    """Recovers the watermark from a (potentially attacked) watermarked image array."""
    coeffs_watermarked_image = process_coefficients(image_array, model, level=level)
    dct_watermarked_coeff = apply_dct(coeffs_watermarked_image[0])
    watermark_array = get_watermark(dct_watermarked_coeff, WATERMARK_SIZE)
    return watermark_array

def w2d_embed(host_image_path, watermark_image_path):
    """Embeds the watermark and returns the original and watermarked host arrays."""
    # 1. Load and Prepare Images
    image_array = convert_image(host_image_path, HOST_SIZE)
    watermark_array = convert_image(watermark_image_path, WATERMARK_SIZE)

    # 2. DWT Decomposition
    coeffs_image = process_coefficients(image_array, MODEL, level=LEVEL)

    # 3. DCT on LL sub-band
    dct_array = apply_dct(coeffs_image[0])

    # 4. Embed Watermark (Modifies dct_array in-place)
    dct_array = embed_watermark(watermark_array, dct_array)

    # 5. Inverse DCT
    coeffs_image[0] = inverse_dct(dct_array)

    # 6. Inverse DWT (Reconstruction)
    image_array_H = pywt.waverec2(coeffs_image, MODEL)

    # 7. Save Final Image
    print_image_from_array(image_array_H, 'image_with_watermark.jpg')

    return image_array, image_array_H, watermark_array # Return all three arrays


# =========================================================================
# === 2. ROBUSTNESS ATTACK FUNCTIONS
# =========================================================================

def attack_jpeg_compression(image_array, quality):
    img = Image.fromarray(np.uint8(image_array.clip(0, 255)))
    attack_path = f'./attacks/jpeg_q{quality}.jpg'
    img.save(attack_path, 'jpeg', quality=quality)
    attacked_img = Image.open(attack_path).convert('L')
    return np.array(attacked_img.getdata()).reshape(image_array.shape)

def attack_gaussian_noise(image_array, sigma):
    normalized_array = image_array / 255.0
    noise = np.random.normal(0, sigma, image_array.shape)
    attacked_array = normalized_array + noise
    attacked_array = np.clip(attacked_array, 0, 1) * 255.0
    Image.fromarray(np.uint8(attacked_array.clip(0, 255))).save(f'./attacks/noise_s{sigma:.2f}.jpg')
    return attacked_array

def attack_gaussian_blur(image_array, sigma):
    attacked_array = gaussian_filter(image_array, sigma=sigma)
    Image.fromarray(np.uint8(attacked_array.clip(0, 255))).save(f'./attacks/blur_s{sigma:.1f}.jpg')
    return attacked_array


# =========================================================================
# === 3. METRIC FUNCTIONS
# =========================================================================

def measure_psnr(original_host, watermarked_host):
    """Calculates PSNR (in dB) to measure Imperceptibility."""
    # PSNR needs inputs to be normalized/standardized, but skimage PSNR handles float/uint8 inputs.
    # We use the float arrays directly from the embedding process.
    # Data range for 8-bit image is 255.
    return PSNR(original_host, watermarked_host, data_range=255)

def measure_ncc(original_wm, recovered_wm):
    """Calculates NCC (Pearson's r) to measure Robustness."""
    W = original_wm.flatten()
    W_prime = recovered_wm.flatten()
    # np.corrcoef returns the correlation matrix
    ncc_value = np.corrcoef(W, W_prime)[0, 1]
    return ncc_value


# =========================================================================
# === 4. EXECUTION AND REPORTING
# =========================================================================

def evaluate_watermarking_scheme():
    """Performs embedding, measures imperceptibility, runs attacks, and measures robustness."""
    print("--- WATERMARKING EVALUATION STARTING ---")

    try:
        # Step 1: Embedding
        original_host_arr, wm_host_arr, original_wm_arr = w2d_embed(IMAGE_HOST, IMAGE_WATERMARK)
    except FileNotFoundError as e:
        print(f"\nFATAL ERROR: {e}")
        print(f"Please ensure '{IMAGE_HOST}' and '{IMAGE_WATERMARK}' are in the current directory.")
        return

    # --- IMPERCEPTIBILITY SCORE ---
    print("\n" + "="*50)
    print("                 IMPERCEPTIBILITY TEST")
    print("="*50)

    psnr_score = measure_psnr(original_host_arr, wm_host_arr)
    print(f"| PSNR (Original vs. Watermarked): {psnr_score:.2f} dB")

    # Interpretation of PSNR
    if psnr_score >= 40:
        print("| Conclusion: EXCELLENT imperceptibility (watermark is highly invisible).")
    elif psnr_score >= 30:
        print("| Conclusion: GOOD imperceptibility (watermark is barely noticeable).")
    else:
        print("| Conclusion: POOR imperceptibility (watermark is likely visible).")

    # --- ROBUSTNESS SCORE ---
    print("\n" + "="*50)
    print("                 ROBUSTNESS TEST (NCC)")
    print("="*50)

    test_results = {}

    # Baseline (No Attack)
    recovered_wm_baseline = recover_watermark(wm_host_arr)
    ncc_baseline = measure_ncc(original_wm_arr, recovered_wm_baseline)
    test_results['Baseline'] = ncc_baseline
    print(f"| {'Baseline (No Attack):':<30} | NCC: {ncc_baseline:.4f} | (Should be close to 1.0)")
    print("-" * 50)

    # Attacks
    attacks_to_run = {
        'JPEG Q=60': ('jpeg', 60),
        'JPEG Q=30': ('jpeg', 30),
        'Noise S=0.10': ('noise', 0.10),
        'Noise S=0.20': ('noise', 0.20),
        'Blur S=1.0': ('blur', 1.0),
        'Blur S=2.0': ('blur', 2.0),
    }

    for name, (attack_type, strength) in attacks_to_run.items():
        if attack_type == 'jpeg':
            attacked_img_arr = attack_jpeg_compression(wm_host_arr, strength)
        elif attack_type == 'noise':
            attacked_img_arr = attack_gaussian_noise(wm_host_arr, strength)
        elif attack_type == 'blur':
            attacked_img_arr = attack_gaussian_blur(wm_host_arr, strength)

        recovered_wm = recover_watermark(attacked_img_arr)
        ncc_score = measure_ncc(original_wm_arr, recovered_wm)
        test_results[name] = ncc_score
        print(f"| {name:<30} | NCC: {ncc_score:.4f} |")

    # Final Robustness Conclusion
    average_ncc = np.mean([ncc for name, ncc in test_results.items() if name != 'Baseline'])
    print("-" * 50)
    print(f"| {'Average NCC (Attack):':<30} | NCC: {average_ncc:.4f} |")

    if average_ncc >= 0.8:
        print("| Conclusion: HIGHLY ROBUST (watermark survives most common attacks).")
    elif average_ncc >= 0.6:
        print("| Conclusion: MODERATELY ROBUST (watermark survives light attacks).")
    else:
        print("| Conclusion: POOR ROBUSTNESS (watermark is easily destroyed).")

    print("\n--- EVALUATION COMPLETE ---")

# Run the full evaluation
evaluate_watermarking_scheme()

--- WATERMARKING EVALUATION STARTING ---

                 IMPERCEPTIBILITY TEST
| PSNR (Original vs. Watermarked): 30.01 dB
| Conclusion: GOOD imperceptibility (watermark is barely noticeable).

                 ROBUSTNESS TEST (NCC)
| Baseline (No Attack):          | NCC: 1.0000 | (Should be close to 1.0)
--------------------------------------------------
| JPEG Q=60                      | NCC: 0.9080 |
| JPEG Q=30                      | NCC: 0.8375 |
| Noise S=0.10                   | NCC: 0.8395 |
| Noise S=0.20                   | NCC: 0.6733 |
| Blur S=1.0                     | NCC: 0.9989 |
| Blur S=2.0                     | NCC: 0.9786 |
--------------------------------------------------
| Average NCC (Attack):          | NCC: 0.8726 |
| Conclusion: HIGHLY ROBUST (watermark survives most common attacks).

--- EVALUATION COMPLETE ---
